# `aidp-object-storage` live test — direct read/write `oci://`
Writes a small CSV to OCI Object Storage and reads it back. Auth is implicit via the cluster's IAM identity — no keys.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
oci_path = os.environ['OCI_OS_TEST_PATH']  # e.g. oci://my-bucket@my-ns/aidp-test/roundtrip.csv
data = [('Alice', 30), ('Bob', 35), ('Charlie', 25)]
df = spark.createDataFrame(data, ['name','age'])
(df.write.mode('overwrite').option('header', True).format('csv').save(oci_path))
print('wrote:', oci_path)


In [ ]:
df = spark.read.option('header', True).format('csv').load(os.environ['OCI_OS_TEST_PATH'])
df.show()


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-object-storage',
    'auth': 'oci-implicit-iam',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
